<a href="https://colab.research.google.com/github/ilinkabibic/Tom-and-Jerry-Character-Classification/blob/main/02_klasican_ml_ika.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Klasičan ML — Ika

Priprema: preuzimanje dataset-a, pristup repo-u, učitavanje gotovog train/val/test split-a.

In [1]:
from google.colab import userdata
import os

os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

!kaggle datasets download -d balabaskar/tom-and-jerry-image-classification
!unzip -q tom-and-jerry-image-classification.zip -d data/

Dataset URL: https://www.kaggle.com/datasets/balabaskar/tom-and-jerry-image-classification
License(s): CC0-1.0
100% 435M/435M [00:04<00:00, 109MB/s]



In [2]:
import os
for root, dirs, files in os.walk('data'):
    level = root.replace('data', '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} fajlova)')

data/ (2 fajlova)
  tom_and_jerry/ (0 fajlova)
    tom_and_jerry/ (0 fajlova)
      tom/ (1930 fajlova)
      jerry/ (1240 fajlova)
      tom_jerry_1/ (780 fajlova)
      tom_jerry_0/ (1528 fajlova)


In [3]:
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
username = "ilinkabibic"
repo_name = "Tom-and-Jerry-Character-Classification"

!git clone https://{token}@github.com/{username}/{repo_name}.git
%cd {repo_name}
!git config user.email "bibic.ilinka@gmail.com"
!git config user.name "Ika"
!pwd

Cloning into 'Tom-and-Jerry-Character-Classification'...
remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 9 (delta 1), reused 8 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (9/9), 29.07 MiB | 5.49 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Tom-and-Jerry-Character-Classification
/content/Tom-and-Jerry-Character-Classification


In [4]:
import pandas as pd
split_df = pd.read_csv('/content/Tom-and-Jerry-Character-Classification/data_split.csv')
print(split_df.shape)
print(split_df.head())

(5478, 5)
     filename  tom  jerry    class  split
0  frame0.jpg    0      0  neither  train
1  frame1.jpg    0      0  neither  train
2  frame2.jpg    0      0  neither  train
3  frame3.jpg    0      0  neither  train
4  frame4.jpg    0      0  neither  train


In [5]:
!pwd

/content/Tom-and-Jerry-Character-Classification


## Pokušaj 1 — BGR histogram boja + HOG + Logistic Regression

In [6]:
!pip install scikit-image -q

import cv2
import numpy as np
from skimage.feature import hog
from tqdm import tqdm

IMG_SIZE = 128
folder_map = {'tom': 'tom', 'jerry': 'jerry', 'both': 'tom_jerry_1', 'neither': 'tom_jerry_0'}
base_dir = '/content/data/tom_and_jerry/tom_and_jerry/'

def extract_features(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    hist_features = []
    for i in range(3):
        hist = cv2.calcHist([img], [i], None, [32], [0, 256])
        hist_features.extend(hist.flatten())

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hog_features = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))

    return np.concatenate([hist_features, hog_features])

features = []
for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
    path = base_dir + folder_map[row['class']] + '/' + row['filename']
    features.append(extract_features(path))

X = np.array(features)
print(X.shape)  # (5478, broj_osobina)

100%|██████████| 5478/5478 [00:47<00:00, 114.96it/s]

(5478, 1860)


In [7]:
train_mask = split_df['split'] == 'train'
val_mask = split_df['split'] == 'val'
test_mask = split_df['split'] == 'test'

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]
y_train = split_df.loc[train_mask, 'class']
y_val = split_df.loc[val_mask, 'class']
y_test = split_df.loc[test_mask, 'class']

print(X_train.shape, X_val.shape, X_test.shape)

(3778, 1860) (850, 1860) (850, 1860)


In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

In [9]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

val_pred = clf.predict(X_val_scaled)
print("Validation accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred))

Validation accuracy: 0.43411764705882355
              precision    recall  f1-score   support

        both       0.34      0.38      0.36       115
       jerry       0.37      0.35      0.36       203
     neither       0.41      0.42      0.41       218
         tom       0.54      0.51      0.53       314

    accuracy                           0.43       850
   macro avg       0.41      0.42      0.42       850
weighted avg       0.44      0.43      0.44       850



**Osobine:** HOG (grayscale, 128×128) + histogram boja (BGR, 32 bin-a po kanalu) = 1860 osobina
**Model:** Logistic Regression (class_weight='balanced')

**Validation accuracy:** 0.434 (43.4%) — macro F1: 0.42

**Zaključak:** Slabo, blizu "uvek pogodi najčešću klasu" (~37%). Globalne osobine verovatno ne hvataju dobro likove jer zauzimaju mali deo kadra u odnosu na pozadinu. Prelazimo na HSV histogram + SVM kao poboljšanje.

## Pokušaj 2 — poboljšanje: HSV histogram boja

In [10]:
def extract_features(path):
    img = cv2.imread(path)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    ranges = [[0, 180], [0, 256], [0, 256]]
    hist_features = []
    for i in range(3):
        hist = cv2.calcHist([hsv], [i], None, [32], ranges[i])
        hist_features.extend(hist.flatten())

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    hog_features = hog(gray, orientations=9, pixels_per_cell=(16, 16), cells_per_block=(2, 2))

    return np.concatenate([hist_features, hog_features])

In [11]:
features = []
for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
    path = base_dir + folder_map[row['class']] + '/' + row['filename']
    features.append(extract_features(path))

X = np.array(features)
print(X.shape)

100%|██████████| 5478/5478 [01:14<00:00, 73.32it/s] 

(5478, 1860)


In [12]:
X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_train_scaled, y_train)

val_pred = clf.predict(X_val_scaled)
print("Validation accuracy:", accuracy_score(y_val, val_pred))
print(classification_report(y_val, val_pred))

Validation accuracy: 0.4364705882352941
              precision    recall  f1-score   support

        both       0.40      0.48      0.43       115
       jerry       0.34      0.30      0.32       203
     neither       0.42      0.46      0.44       218
         tom       0.52      0.50      0.51       314

    accuracy                           0.44       850
   macro avg       0.42      0.43      0.42       850
weighted avg       0.44      0.44      0.44       850



### SVM umesto Logistic Regression

Probano jer je granica između klasa verovatno nelinearna.

In [13]:
from sklearn.svm import SVC

svm_clf = SVC(kernel='rbf', class_weight='balanced')
svm_clf.fit(X_train_scaled, y_train)

val_pred_svm = svm_clf.predict(X_val_scaled)
print("Validation accuracy (SVM):", accuracy_score(y_val, val_pred_svm))
print(classification_report(y_val, val_pred_svm))

Validation accuracy (SVM): 0.48705882352941177
              precision    recall  f1-score   support

        both       0.56      0.57      0.57       115
       jerry       0.39      0.20      0.27       203
     neither       0.40      0.56      0.47       218
         tom       0.57      0.59      0.58       314

    accuracy                           0.49       850
   macro avg       0.48      0.48      0.47       850
weighted avg       0.48      0.49      0.47       850



In [14]:
import numpy as np

np.save('/content/Tom-and-Jerry-Character-Classification/features_hog_color.npy', X.astype(np.float32))

%cd /content/Tom-and-Jerry-Character-Classification
!git add features_hog_color.npy
!git commit -m "add extracted HOG+HSV histogram features (final baseline)"
!git push

/content/Tom-and-Jerry-Character-Classification
On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
Everything up-to-date
